In [ ]:
import os

from aott.PSF_Processing import PSF_Processing
from aott.Atmosphere_Characterization import Atmosphere_Characterization
from aott.AnalysisViewer import AnalysisViewer
from pathlib import Path
import subprocess
import numpy as np
import pylab as plt

In [ ]:
latest_file = r"C:\Users\foyarzun\Nextcloud\AOTelemetryToolbox\data\simulated_data_r0m_0.04_V0mps_4.61_L0m_25.00_tau0ms_2.69.hdf5"

In [ ]:
p2 = PSF_Processing(latest_file, 0.3)
p2.elevation = 90
p2.SetPSFModel()
p2.AnalyzeAllTheFile()

In [ ]:
plt.imshow(p2.long_exp)

In [ ]:
atm_char = Atmosphere_Characterization(latest_file, batch_duration=0.5)
atm_char.AnalyzeAllTheFile()

In [ ]:
av = AnalysisViewer(latest_file)

av.CreateAtmosphericAnalysisFigures()
av.CreatePSFAnalysisFigures()
av.SaveFigureManifest()
av.RemoveFigureFiles()

In [ ]:
import subprocess
from datetime import datetime

DATE = "/" + datetime.now().strftime("%Y-%m-%d")
file_title = str(latest_file).split("/")[-1].split(".hdf5")[0]
target_name = file_title.split("-")[0]

cmd = [
    "typst",
    "compile",
    "ao_report.typ",
    "ao_report" + file_title + ".pdf",
    "--input",
    "AOtitle=" + str(latest_file).split("/")[-1].split(".hdf5")[0],
    "--input",
    "telescope=T152-Papyrus",
    "--input",
    "date=" + DATE,
    "--input",
    "target=" + target_name,
    "--input",
    f"elevation={p2.elevation:.1f}",
    "--input",
    "loop_gain=" + str(atm_char.loop_gain),
    "--input",
    "loop_leak=" + str(atm_char.loop_leak),
    "--input",
    "loop_freq=" + str(atm_char.freq),
]

In [ ]:
try:
    result = subprocess.run(
        cmd,
        check=True,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("STDOUT:")
    print(e.stdout)
    print("\nSTDERR:")
    print(e.stderr)